# Can a linear map read the location from a Whisper embedding?

A feasibility test: is geography (and so, presumably, dialect) *linearly* encoded in Whisper's encoder, and in how
many dimensions? For every postal code (PLZ) $k$ with clips $\mathcal{I}_k$ we take the centroid of its clip
embeddings at layer $\ell$ of model $\theta$,

$$p_k = \frac{1}{n_k}\sum_{i\in\mathcal{I}_k} e_i^{(\theta,\ell)}, \qquad P = [p_1, \dots, p_N]^\top \in \mathbb{R}^{N\times d},$$

and the PLZ's coordinates in LV95 (km), centred and divided by one shared scale $s$ (so both axes keep their
proportions and errors convert back to km), $L \in \mathbb{R}^{N\times 2}$. The probe is ridge regression on the
standardised columns of $P$,

$$W^\star = \arg\min_{W\in\mathbb{R}^{d\times 2}} \lVert PW - L\rVert_F^2 + \lambda\lVert W\rVert_F^2 = (P^\top P + \lambda I)^{-1}P^\top L,$$

with $\lambda$ chosen inside each training fold. With $d \ge N$ (1280 dimensions, ~200 PLZs) plain least squares
would fit any $L$ exactly, so the regularisation and the held-out evaluation carry the whole result.

1. **Data**: clips, PLZs and their coordinates; how many speakers stand behind a PLZ.
2. **Embeddings**: precomputed files, or any Whisper checkpoint (pretrained, randomly initialised, or our
   fine-tuned M3 `best.pt`) embedded now.
3. **Layer sweep**: held-out error in km per model and layer, under PLZ-disjoint folds (*random*) and
   canton-disjoint folds (*canton*, no neighbouring PLZ to interpolate from), against the mean predictor.
4. **Is it real?** Permutation test of the best layer per model, and the predictions on the map.
5. **Few dimensions?** Group lasso (error vs number of dimensions used), PCA rank, single-dimension correlations.
6. **Clip-level probe**: the same map fitted on clips instead of centroids.

In [ ]:
from pathlib import Path

import geopandas as gpd
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.linear_model import MultiTaskLasso, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm, trange

from maloney.data.meta import read_h5_index, read_tsv
from maloney.embeddings import EmbeddingStore

ROOT = Path(rootutils.find_root(indicator=".project-root"))
# every PLZ filled up to >= 3 female and 3 male speakers, 40 clips each (uv run anonymize-zip)
DATA_DIR = ROOT / "data/stt4sg-350_synth_zip"
SEED = 0

# Clip metadata (speaker, zip code) and the zip code table of scripts/build_zip_geo.py.
TSVS = [DATA_DIR / f"{name}.tsv" for name in ("train_all", "valid", "test")]
GEO_TABLE = DATA_DIR / "zip_municipality.csv"
UNIT = "zipcode"  # what gets a centroid: "zipcode" (PLZ) or "bfs_nr" (municipality)
CLIPS_PER_SPEAKER = 40  # random clips per speaker, the same for every model (None: all; needs precomputed files)
MIN_CLIPS = 10  # PLZs with fewer clips are dropped
SPEAKER_FIRST = True  # centroid = mean of the speakers' means (else of all clips: talkative speakers dominate)

# name -> source. A source is
#   - a precomputed .h5 of scripts/extract_whisper_embeddings.py,
#   - a Whisper checkpoint on the HF Hub ("openai/whisper-small"), embedded now,
#   - "random:<checkpoint>": that architecture with random weights (does training matter at all?),
#   - an M3 checkpoint (logs/train/<run>/.../checkpoints/best.pt): the fine-tuned encoder.
# Sources embedded now need the clips' audio (audio_16k.h5) and are cached in CACHE_DIR.
MODELS = {
    "tiny": "openai/whisper-tiny",
    # "large-v3": DATA_DIR / "embeddings/openai__whisper-large-v3.h5",
    # "base": "openai/whisper-base",
    # "small": "openai/whisper-small",
    # "medium": "openai/whisper-medium",
    # "random-tiny": "random:openai/whisper-tiny",
    # "m3": ROOT / "logs/train/m3/runs/<date>/checkpoints/best.pt",
}
CACHE_DIR = DATA_DIR / "embeddings/notebook"

N_FOLDS = 5
ALPHAS = np.logspace(-1, 6, 29)  # ridge penalties searched inside every training fold
N_PERMUTATIONS = 200

## 1. Data

Every clip gets its speaker's PLZ and the PLZ's centroid in LV95 (`zip_E`/`zip_N`, or the municipality's for
`UNIT = "bfs_nr"`). The speaker subsample (`CLIPS_PER_SPEAKER`) is drawn once, so all models see the same clips.

Keep the last table in mind: if most PLZs have a single speaker, a PLZ centroid is a **speaker** embedding, and
the probe can also pick up voice, gender, age or microphone wherever they happen to correlate with place.

In [ ]:
COORDS = {"zipcode": ("zip_E", "zip_N"), "bfs_nr": ("muni_E", "muni_N")}[UNIT]
geo = pd.read_csv(GEO_TABLE, dtype={"zipcode": str}).dropna(subset=["bfs_nr", *COORDS])
geo = geo.astype({"bfs_nr": int}).rename(columns={"canton": "unit_canton", COORDS[0]: "E", COORDS[1]: "N"})

meta = pd.concat([read_tsv(tsv) for tsv in TSVS if tsv.is_file()]).drop_duplicates("path")
n_all = len(meta)
meta = meta.merge(geo[["zipcode", "bfs_nr", "municipality", "unit_canton", "E", "N"]], on="zipcode", how="inner")
print(f"{len(meta)} of {n_all} clips have a located zip code")
if UNIT == "bfs_nr":  # a municipality's coordinates are the same for all its zip codes
    meta[["E", "N"]] = meta.groupby("bfs_nr")[["E", "N"]].transform("first")
if CLIPS_PER_SPEAKER is not None:
    meta = meta.sample(frac=1, random_state=SEED).groupby("client_id").head(CLIPS_PER_SPEAKER)
meta = meta.sort_values("path").reset_index(drop=True)

meta.groupby(UNIT).client_id.nunique().value_counts().sort_index().rename("PLZs").rename_axis("speakers per PLZ").to_frame().T

## 2. Embeddings

A source gives the clips it covers (`.meta`, with `frames`, the encoder frames each clip embedding averages) and
`load_layer(layer)`, the clip embeddings of one layer row-aligned with them. Layers are read one at a time
(a layer of whisper-large-v3 over the whole release is ~1.2 GB).

For an M3 checkpoint, the encoder is truncated at its `pool_layer` as in training, so the layers above it do not
exist; random weights use the architecture's own initialisation (seeded).

In [ ]:
class Precomputed:
    def __init__(self, path, meta):
        store = EmbeddingStore(path)
        self.path, self.layers, self.dim = Path(path), store.layers, store.dim
        self.meta = meta[meta.path.isin(store.row.index)].reset_index(drop=True)
        self.rows = store.rows(self.meta.path)
        self.meta["frames"] = store.frames[self.rows]

    def load_layer(self, layer):
        with h5py.File(self.path, "r") as f:
            return f[f"layer_{layer:02d}"][:][self.rows].astype(np.float32)


def make_embedder(source, device):
    import torch
    from transformers import WhisperConfig, WhisperModel

    from maloney.embeddings.whisper import WhisperEmbedder, default_layers

    if source.endswith(".pt"):
        state = torch.load(source, map_location="cpu", weights_only=False)
        name, pool_layer = state["config"]["model"]["name"], state["config"]["model"]["pool_layer"]
        n_layers = WhisperConfig.from_pretrained(name).encoder_layers
        pool_layer = pool_layer or n_layers
        embedder = WhisperEmbedder(name, [l for l in default_layers(n_layers) if l <= pool_layer], device)
        encoder = embedder.encoder
        if pool_layer < n_layers:  # as in WhisperGeoClassifier
            encoder.layers = encoder.layers[:pool_layer]
            encoder.layer_norm = torch.nn.Identity()
            encoder.config.encoder_layers = pool_layer
        encoder.load_state_dict({k.removeprefix("encoder."): v for k, v in state["model"].items()
                                 if k.startswith("encoder.")})
    elif source.startswith("random:"):
        name = source.removeprefix("random:")
        embedder = WhisperEmbedder(name, device=device)
        torch.manual_seed(SEED)
        embedder.encoder = (WhisperModel(WhisperConfig.from_pretrained(name)).get_encoder()
                            .to(embedder.device, embedder.dtype).eval())
    else:
        embedder = WhisperEmbedder(source, device=device)
    return embedder


class Embedded:
    """Embeds the clips of `meta` now (cached per source and clip set)."""

    def __init__(self, source, meta, batch_size=16):
        source = str(source)
        self.meta = meta.copy()
        # an M3 checkpoint is named after its run directory (.../<run>/checkpoints/best.pt)
        slug = f"m3_{Path(source).parents[1].name}" if source.endswith(".pt") else source.replace("/", "__").replace(":", "_")
        cache = CACHE_DIR / f"{slug}_{len(meta)}.npz"
        if cache.is_file() and (saved := np.load(cache, allow_pickle=True))["path"].tolist() == meta.path.tolist():
            self.emb, self.meta["frames"], self.layers = saved["emb"], saved["frames"], saved["layers"].tolist()
        else:
            import torch

            device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
            embedder = make_embedder(source, device)
            index, _ = read_h5_index(DATA_DIR / "audio_16k.h5")
            loc = index.loc[meta.path]
            embs, frames = [], []
            with h5py.File(DATA_DIR / "audio_16k.h5", "r") as f:
                for i in trange(0, len(meta), batch_size, desc=source[-40:]):
                    audios = [f["audio"][o: o + n].astype(np.float32) / 32767
                              for o, n in zip(loc.offset[i: i + batch_size], loc.length[i: i + batch_size])]
                    e, fr = embedder.embed(audios)
                    embs.append(e.astype(np.float16))
                    frames.append(fr)
            self.emb, self.meta["frames"], self.layers = np.concatenate(embs), np.concatenate(frames), embedder.layers
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            np.savez(cache, emb=self.emb, frames=self.meta.frames.to_numpy(), layers=self.layers,
                     path=meta.path.to_numpy())
        self.dim = self.emb.shape[-1]

    def load_layer(self, layer):
        return self.emb[:, self.layers.index(layer)].astype(np.float32)


def load_source(source, meta):
    return Precomputed(source, meta) if str(source).endswith(".h5") else Embedded(source, meta)


sources = {name: load_source(source, meta) for name, source in MODELS.items()}
for name, src in sources.items():
    print(f"{name:>10}: {len(src.meta):6d} of {len(meta)} clips, dim {src.dim}, layers {src.layers}")

The PLZs used are those with at least `MIN_CLIPS` clips **in every source**, so all models are scored on the same
PLZs and folds. A precomputed file that covers only part of the clips (e.g. one split) shrinks this set for all.

In [ ]:
counts = pd.concat([src.meta.groupby(UNIT).size().rename(name) for name, src in sources.items()], axis=1)
keep = counts.index[(counts.fillna(0) >= MIN_CLIPS).all(axis=1)]

units = meta[meta[UNIT].isin(keep)].groupby(UNIT).agg(
    E=("E", "first"), N=("N", "first"), municipality=("municipality", "first"), canton=("unit_canton", "first"),
    dialect_region=("dialect_region", lambda r: r.mode().iloc[0]), speakers=("client_id", "nunique"),
    clips=("path", "size"))
REGIONS = sorted(units.dialect_region.unique())
REGION_COLORS = dict(zip(REGIONS, plt.get_cmap("tab10").colors))

# Targets: LV95 in km, centred, one shared scale (root mean squared distance to the centre).
C = units[["E", "N"]].to_numpy() / 1000
C_MEAN = C.mean(0)
S = np.sqrt(((C - C_MEAN) ** 2).sum(1).mean())
L = (C - C_MEAN) / S

# Fold of every PLZ: random (PLZ-disjoint) and canton-disjoint.
FOLDS = {
    "random": np.zeros(len(units), int),
    "canton": np.zeros(len(units), int),
}
for k, (_, test) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(units)):
    FOLDS["random"][test] = k
for k, (_, test) in enumerate(GroupKFold(N_FOLDS).split(units, groups=units.canton)):
    FOLDS["canton"][test] = k

print(f"{len(units)} PLZs, {units.speakers.sum()} speakers, {units.clips.sum()} clips; scale s = {S:.1f} km")
units.groupby("dialect_region").agg(PLZs=("E", "size"), speakers=("speakers", "sum"),
                                    cantons=("canton", lambda c: ", ".join(sorted(set(c)))))

## 3. Layer sweep

`centroids` builds $P$: with `SPEAKER_FIRST`, the frame-weighted mean of each speaker's clips, then the plain mean
over the speakers of a PLZ. `probe` standardises $P$ and fits `RidgeCV` (efficient leave-one-out over `ALPHAS`)
on the training PLZs of each fold, so $\lambda$ never sees the held-out PLZs. Errors are Euclidean distances in
LV95, back in km; $R^2$ is per axis (E, N) on the held-out predictions.

In [ ]:
def pool(X, groups, weights=None):
    """Weighted mean of the rows of X per group: (sorted group keys, [n_groups, dim])."""
    keys, codes = np.unique(np.asarray(groups), return_inverse=True)
    w = np.ones(len(X)) if weights is None else np.asarray(weights, dtype=float)
    sums = np.zeros((len(keys), X.shape[1]))
    np.add.at(sums, codes, X * w[:, None])
    return keys, sums / np.bincount(codes, w)[:, None]


def centroids(X, sub, speaker_first=SPEAKER_FIRST):
    """P [len(units), dim], rows in the order of `units`, from clip embeddings X row-aligned with `sub`."""
    if speaker_first:
        speaker_ids, X = pool(X, sub.client_id, sub.frames)
        keys, P = pool(X, sub.groupby("client_id")[UNIT].first().loc[speaker_ids])
    else:
        keys, P = pool(X, sub[UNIT], sub.frames)
    return pd.DataFrame(P, index=keys).loc[units.index].to_numpy()


def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))


def probe(P, fold, Y=None, make=ridge):
    """Held-out predictions [N, 2] of Y (default L) from P, one model per fold."""
    Y = L if Y is None else Y
    pred = np.zeros_like(Y)
    for k in np.unique(fold):
        train, test = fold != k, fold == k
        pred[test] = make().fit(P[train], Y[train]).predict(P[test])
    return pred


def errors_km(pred, Y=None):
    return np.linalg.norm(pred - (L if Y is None else Y), axis=1) * S


def scores(pred):
    err = errors_km(pred)
    return {"median_km": np.median(err), "mean_km": err.mean(),
            "r2_E": r2_score(L[:, 0], pred[:, 0]), "r2_N": r2_score(L[:, 1], pred[:, 1])}


class MeanPredictor:
    """Baseline: the training PLZs' mean location."""

    def fit(self, X, Y):
        self.mean = Y.mean(0)
        return self

    def predict(self, X):
        return np.tile(self.mean, (len(X), 1))


BASELINE = {scheme: scores(probe(np.zeros((len(units), 1)), fold, make=MeanPredictor)) for scheme, fold in FOLDS.items()}
pd.DataFrame(BASELINE).T.round(3)

In [ ]:
CENTROIDS = {}  # (model, layer) -> P
rows = []
for name, src in sources.items():
    sub = src.meta[src.meta[UNIT].isin(units.index)]
    mask = src.meta[UNIT].isin(units.index).to_numpy()
    for layer in tqdm(src.layers, desc=name):
        P = centroids(src.load_layer(layer)[mask], sub.reset_index(drop=True))
        CENTROIDS[name, layer] = P
        for scheme, fold in FOLDS.items():
            rows.append({"model": name, "layer": layer, "depth": layer / max(src.layers), "scheme": scheme,
                         **scores(probe(P, fold))})
sweep = pd.DataFrame(rows)
sweep.pivot_table(index=["model", "layer"], columns="scheme", values=["median_km", "r2_E", "r2_N"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for (name, scheme), g in sweep.groupby(["model", "scheme"], sort=False):
    style = dict(marker="o", linestyle="-" if scheme == "random" else "--",
                 color=f"C{list(sources).index(name)}", label=f"{name} ({scheme})")
    axes[0].plot(g.depth, g.median_km, **style)
    axes[1].plot(g.depth, g.r2_E, **style)
    axes[2].plot(g.depth, g.r2_N, **style)
for scheme, ls in [("random", ":"), ("canton", "-.")]:
    axes[0].axhline(BASELINE[scheme]["median_km"], color="grey", linestyle=ls, label=f"mean predictor ({scheme})")
for ax, title in zip(axes, ["held-out median error (km)", "$R^2$ east", "$R^2$ north"]):
    ax.set(title=title, xlabel="relative depth (layer / last layer)")
axes[1].axhline(0, color="grey", linestyle=":")
axes[2].axhline(0, color="grey", linestyle=":")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

BEST = sweep[sweep.scheme == "random"].loc[lambda d: d.groupby("model").median_km.idxmin()].set_index("model").layer
print("best layer per model (random folds):", BEST.to_dict())

## 4. Is it real?

**Permutation test**: shuffle which PLZ gets which coordinates and rerun the whole probe (including the choice
of $\lambda$). The p-value is the share of shuffles that do at least as well as the real coordinates. The best
layer was itself picked on these folds, which favours the real fit slightly; the canton folds and a large gap to
the null are the more convincing evidence.

In [ ]:
rng = np.random.default_rng(SEED)
perm = {}
for name, layer in BEST.items():
    P = CENTROIDS[name, layer]
    real = np.median(errors_km(probe(P, FOLDS["random"])))
    null = []
    for _ in trange(N_PERMUTATIONS, desc=name, leave=False):
        Y = L[rng.permutation(len(L))]
        null.append(np.median(errors_km(probe(P, FOLDS["random"], Y), Y)))
    perm[name] = {"layer": layer, "median_km": real, "null_median_km": np.median(null),
                  "p": (1 + np.sum(np.array(null) <= real)) / (1 + N_PERMUTATIONS), "null": np.array(null)}

fig, axes = plt.subplots(1, len(perm), figsize=(5 * len(perm), 3.2), squeeze=False)
for ax, (name, r) in zip(axes[0], perm.items()):
    ax.hist(r["null"], bins=30, color="lightgrey")
    ax.axvline(r["median_km"], color="C3", label=f"real ({r['median_km']:.1f} km)")
    ax.set(title=f"{name}, layer {r['layer']}: p = {r['p']:.3f}", xlabel="median error (km), shuffled coordinates")
    ax.legend()
fig.tight_layout()
plt.show()

The held-out predictions of the best model and layer on the map (random folds): an arrow from each PLZ's true
location to its prediction, coloured by dialect region. Shrinkage towards the centre is expected from ridge; what
matters is whether the predictions keep the regions apart and in the right order. On the right, the error
against the number of speakers behind a PLZ.

In [ ]:
best_name = min(perm, key=lambda n: perm[n]["median_km"])
best_layer = BEST[best_name]
pred = probe(CENTROIDS[best_name, best_layer], FOLDS["random"])
P_km = pred * S + C_MEAN
cantons = gpd.read_file(ROOT / "data/geo/cantons.geojson").to_crs(2056)
cantons.geometry = cantons.geometry.scale(1e-3, 1e-3, origin=(0, 0))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
cantons.boundary.plot(ax=ax, color="lightgrey", linewidth=0.6)
colors = [REGION_COLORS[r] for r in units.dialect_region]
for (x0, y0), (x1, y1), c in zip(C, P_km, colors):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="->", color=c, alpha=0.6, lw=0.8))
ax.scatter(*C.T, c=colors, s=12, edgecolor="k", linewidth=0.3)
ax.scatter(*P_km.T, c=colors, s=10, marker="x")
ax.scatter(*C_MEAN, c="k", marker="+", s=100)
ax.legend(handles=[plt.Line2D([], [], color=c, marker="o", linestyle="", label=r) for r, c in REGION_COLORS.items()],
          fontsize=8, loc="upper left")
ax.set(title=f"{best_name}, layer {best_layer}: true (o) -> predicted (x), held-out", aspect="equal",
       xticks=[], yticks=[])

err = errors_km(pred)
axes[1].scatter(units.speakers + rng.uniform(-0.15, 0.15, len(units)), err, c=colors, s=12)
axes[1].set(xscale="log", xlabel="speakers per PLZ", ylabel="held-out error (km)", title="error vs speakers")
fig.tight_layout()
plt.show()
pd.DataFrame({"error_km": err, "dialect_region": units.dialect_region.to_numpy()}).groupby("dialect_region").error_km.describe()[["count", "50%", "mean"]].round(1)

## 5. A few dimensions, or spread out?

Ridge says *whether* the location is linearly decodable, not *where* it sits. Three views, on the best layer of
each model:

- **Group lasso** (`MultiTaskLasso`): $\min_W \lVert PW-L\rVert_F^2 + \alpha\sum_j \lVert W_{j,:}\rVert_2$ drops
  whole dimensions. Along the $\alpha$ path: held-out error against the number of dimensions kept (counted on
  the fit to all PLZs). If a handful of dimensions get close to ridge, the information is concentrated
  *in the basis*.
- **PCA rank**: ridge on the top $r$ principal components (PCA fitted per fold). Early saturation means a
  low-dimensional *subspace*, not necessarily aligned with individual dimensions.
- **Single dimensions**: Pearson correlation of every dimension with east and north, against the largest
  correlation under shuffled coordinates. A heavy tail above that line would mean "dialect dimensions".

In [ ]:
def lasso_path(P, fold, n_alphas=16):
    Z = StandardScaler().fit_transform(P)
    alpha_max = np.sqrt(((Z.T @ L) ** 2).sum(1)).max() / len(Z)  # smallest alpha that keeps no dimension
    out = []
    for alpha in alpha_max * np.logspace(-0.05, -2.5, n_alphas):
        make = lambda: make_pipeline(StandardScaler(), MultiTaskLasso(alpha=alpha, max_iter=5000, tol=1e-3))
        active = (np.abs(make().fit(P, L)[-1].coef_).sum(0) > 0).sum()
        out.append({"alpha": alpha, "dims": active, "median_km": np.median(errors_km(probe(P, fold, make=make)))})
    return pd.DataFrame(out)


def pca_curve(P, fold, ranks=(1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128)):
    ranks = [r for r in ranks if r < (fold != 0).sum()]
    make = lambda r: lambda: make_pipeline(StandardScaler(), PCA(r, random_state=SEED), RidgeCV(alphas=ALPHAS))
    return pd.DataFrame([{"rank": r, "median_km": np.median(errors_km(probe(P, fold, make=make(r))))} for r in ranks])


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
lasso, pca = {}, {}
for i, (name, layer) in enumerate(tqdm(BEST.items(), desc="models")):
    P = CENTROIDS[name, layer]
    lasso[name], pca[name] = lasso_path(P, FOLDS["random"]), pca_curve(P, FOLDS["random"])
    ridge_km = sweep.query("model == @name and layer == @layer and scheme == 'random'").median_km.item()
    for ax, df, x in [(axes[0], lasso[name], "dims"), (axes[1], pca[name], "rank")]:
        ax.plot(df[x], df.median_km, marker="o", color=f"C{i}", label=f"{name}, layer {layer}")
        ax.axhline(ridge_km, color=f"C{i}", linestyle=":")
for ax, title, xlabel in [(axes[0], "group lasso: error vs dimensions kept", "dimensions with non-zero weight"),
                          (axes[1], "ridge on top-r principal components", "r")]:
    ax.axhline(BASELINE["random"]["median_km"], color="grey", linestyle="--", label="mean predictor")
    ax.set(title=title, xlabel=xlabel, ylabel="held-out median error (km)", xscale="log")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
def dimension_correlations(P, n_null=50):
    """|r| of every (standardised) dimension with E and N, and the max |r| per shuffle of the PLZs."""
    Z = StandardScaler().fit_transform(P)
    corr = lambda Y: np.abs(np.array([[pearsonr(Z[:, j], Y[:, a]).statistic for a in (0, 1)] for j in range(Z.shape[1])]))
    null = [corr(L[rng.permutation(len(L))]).max(0) for _ in range(n_null)]
    return corr(L), np.array(null)


fig, axes = plt.subplots(1, len(BEST), figsize=(6 * len(BEST), 3.5), squeeze=False)
top_dims = {}
for ax, (name, layer) in zip(axes[0], BEST.items()):
    r, null = dimension_correlations(CENTROIDS[name, layer])
    for a, axis in enumerate("EN"):
        ax.plot(np.sort(r[:, a])[::-1], label=f"|r| with {axis}", color=f"C{a}")
        ax.axhline(np.quantile(null[:, a], 0.95), color=f"C{a}", linestyle=":", label=f"95% of max |r|, shuffled ({axis})")
    ax.set(title=f"{name}, layer {layer}", xlabel="dimension (sorted)", xscale="log", ylabel="|Pearson r|")
    ax.legend(fontsize=7)
    top_dims[name] = pd.DataFrame(r, columns=["abs_r_E", "abs_r_N"]).assign(max=lambda d: d.max(1)).nlargest(8, "max")
fig.tight_layout()
plt.show()
pd.concat(top_dims, names=["model", "dimension"]).round(3)

## 6. Clip-level probe

The same linear map fitted on clips: every clip of a training PLZ gets its PLZ's coordinates, and a held-out PLZ's
prediction is the mean over its clips. Many more samples than PLZs, but each one noisier; if this beats the
centroid probe, averaging before fitting throws away usable variation. Same folds as above.

In [ ]:
clip_rows = []
for name, layer in BEST.items():
    src = sources[name]
    mask = src.meta[UNIT].isin(units.index).to_numpy()
    sub = src.meta[mask].reset_index(drop=True)
    X = src.load_layer(layer)[mask]
    unit_pos = pd.Series(np.arange(len(units)), index=units.index).loc[sub[UNIT]].to_numpy()
    for scheme, fold in FOLDS.items():
        clip_fold, pred = fold[unit_pos], np.zeros((len(units), 2))
        for k in np.unique(fold):
            train = clip_fold != k
            model = ridge().fit(X[train], L[unit_pos[train]])
            test_units = np.flatnonzero(fold == k)
            per_clip = model.predict(X[~train])
            _, pred[test_units] = pool(per_clip, unit_pos[~train])
        centroid_km = sweep.query("model == @name and layer == @layer and scheme == @scheme").median_km.item()
        clip_rows.append({"model": name, "layer": layer, "scheme": scheme, "centroid_median_km": centroid_km,
                          **{f"clip_{k}": v for k, v in scores(pred).items()}})
pd.DataFrame(clip_rows).round(3)

## Summary

Best layer per model (picked on random folds), its error under both fold schemes, the mean predictor and the
permutation p-value.

In [ ]:
summary = sweep.pivot_table(index=["model", "layer"], columns="scheme", values="median_km")
summary = summary.loc[list(BEST.items())].reset_index().rename(columns={"random": "median_km_random", "canton": "median_km_canton"})
summary["mean_predictor_random"] = BASELINE["random"]["median_km"]
summary["mean_predictor_canton"] = BASELINE["canton"]["median_km"]
summary["p_permutation"] = summary.model.map({n: r["p"] for n, r in perm.items()})
summary.round(3)